# 一、元类（Metaclass）：创建类的“类”
在 Python 中，一切皆对象—— 包括我们用class定义的类本身。类是用来创建实例对象的 “模板”，而元类（Metaclass）则是用来创建类的 “模板”。
- 元类->创建->类->创建->实例对象
- Python 默认的元类是type，也就是说，我们平时用class定义的所有类，本质上都是type元类的实例。
1. 用type直接创建类（元类的底层逻辑）
    - 通常用class关键字定义类，但实际上type可以直接动态创建类，语法为：
        - type(类名, 父类元组, 类的属性/方法字典)

- 案例1:用type动态创建一个简单类

In [1]:
# 定义一个方法，后续会作为类的方法
def print_hello(self):
    print(f"Hello, {self.name}!")

# 用type创建类：类名是Person，父类为空，包含name属性和print_hello方法
Person = type(
    "Person",  # 类名
    (),        # 父类元组（空表示无父类）
    {
        "name": "Alice",  # 类属性
        "print_hello": print_hello  # 类方法
    }
)

# 测试：创建实例并调用方法
p = Person()
print(p.name)  # 输出：Alice
p.print_hello()  # 输出：Hello, Alice!

Alice
Hello, Alice!


解释：这里Person类不是用class定义的，而是直接通过type元类“制造”出来的，和普通class定义的类完全等价。

2. 自定义元类：控制类的创建过程
    - 我们可以通过继承type来自定义元类，从而在类被创建时自动执行额外逻辑（比如添加属性、修改方法、验证类结构等）。
    - 自定义元类的核心是重写__new__方法（在类创建前调用，负责生成类对象）或__init__方法（在类创建后调用，负责初始化类对象）。
- 案例2:自定义元类，自动给类添加“作者”属性

In [2]:
# 自定义元类：继承type
class AuthorMeta(type):
    # __new__方法：在类创建前调用，负责生成类对象
    def __new__(cls, name, bases, attrs):
        # 自动给类添加一个author属性
        attrs["author"] = "Python大师"
        # 调用父类type的__new__方法，完成类的创建
        return super().__new__(cls, name, bases, attrs)

# 定义类时，通过metaclass指定使用自定义元类
class MyBook(metaclass=AuthorMeta):
    title = "Python进阶指南"

# 测试：查看类的author属性（自动添加的！）
print(MyBook.title)   # 输出：Python进阶指南
print(MyBook.author)  # 输出：Python大师（元类自动添加的属性）

Python进阶指南
Python大师


解释：当我们定义MyBook类时，Python 会用AuthorMeta元类来创建它。AuthorMeta的__new__方法在创建类前，偷偷给类的属性字典里加了author，所以MyBook类天生就有author属性。

# 二、property属性：把方法“伪装”成属性
- property是 Python 中一种特殊的属性，它能将方法的调用伪装成普通属性的访问，从而实现对属性的封装、验证和计算。
- 用了property后，可以像访问属性一样调用方法，不用加()，还能控制属性的 “读、写、删” 权限。
1. 两种使用方法
    - 用property()函数（传统方式）
    - 用@property装饰器（更简洁，推荐）
2. 案例1:用property()函数控制属性访问
    - 假设我们有一个Person类，想控制age属性的范围（比如 0-120 岁），可以用property()绑定 getter、setter 方法。

In [3]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self._age = age  # 用下划线开头，表示“私有”属性（约定俗成）

    # getter方法：获取_age的值
    def get_age(self):
        return self._age

    # setter方法：设置_age的值（加验证）
    def set_age(self, value):
        if 0 < value < 120:
            self._age = value
        else:
            raise ValueError("年龄必须在0-120之间！")

    # 用property()绑定getter和setter，生成age属性
    age = property(get_age, set_age)

# 测试
p = Person("Bob", 25)
print(p.age)  # 输出：25（像访问属性一样，实际调用get_age()）

p.age = 30    # 像修改属性一样，实际调用set_age(30)
print(p.age)  # 输出：30

# p.age = 150  # 会报错：ValueError: 年龄必须在0-120之间！

25
30


解释：
- _age是真正存储数据的 “私有” 属性（Python 没有真正的私有，靠下划线约定）。
- age = property(get_age, set_age)把get_age和set_age绑定成age属性，访问p.age时自动调get_age，赋值时自动调set_age。

3. 案例2:用@property装饰器（更简洁）
    - @property是 Python 的装饰器语法，能让代码更优雅。直接把 getter 方法用@property装饰，setter 方法用@属性名.setter装饰。

In [4]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius

    # 用@property装饰getter方法，把celsius变成属性
    @property
    def celsius(self):
        return self._celsius

    # 用@celsius.setter装饰setter方法，控制赋值
    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("温度不能低于绝对零度！")
        self._celsius = value

    # 定义fahrenheit属性，动态计算（只有getter，只读）
    @property
    def fahrenheit(self):
        return self._celsius * 9/5 + 32

# 测试
t = Temperature(25)
print(t.celsius)    # 输出：25（访问属性）
print(t.fahrenheit) # 输出：77.0（动态计算，不用加()）

t.celsius = 30      # 修改celsius
print(t.fahrenheit) # 输出：86.0（自动更新）

# t.fahrenheit = 100  # 会报错：AttributeError: can't set attribute（因为没定义setter，只读）

25
77.0
86.0


解释：
- @property装饰celsius方法后，t.celsius就像访问属性一样，实际调用方法。
- @celsius.setter装饰的方法负责处理赋值，加了温度验证。
- fahrenheit只有@property没有 setter，所以是只读属性，不能赋值。